<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/2stage_v2s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/drive')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [ ]:
import os
import shutil
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    ReduceLROnPlateau,
    EarlyStopping
)
#from tensorflow.keras import mixed_precision
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.test.gpu_device_name())

TensorFlow version: 2.20.0
GPU: /device:GPU:0


In [ ]:
BASE = "/content/newdata"

IMG_SRC = "/drive/MyDrive/Colab Notebooks/newdata"

CHECKPOINT_DIR = "/drive/MyDrive/checkpoints"

MODEL_SAVE_PATH = "/drive/MyDrive/Colab Notebooks/Models/dermoscopy/final_model.keras"

In [ ]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)

shutil.copytree(IMG_SRC, BASE)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
batch_size = 16
image_size = 256
num_classes = 2

In [ ]:
augment = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),

    layers.RandomRotation(0.05),

])

In [ ]:
def add_edge_map(image, label):

    image = tf.cast(image, tf.float32)

    # Augmentation
    image = augment(image)

    # EfficientNet preprocessing
    image = preprocess_input(image)

    # Grayscale
    gray = tf.image.rgb_to_grayscale(image)

    # Sobel edges
    sobel = tf.image.sobel_edges(gray)

    edge = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))

    # Normalize safely
    edge = edge / (tf.reduce_max(edge) + 1e-6)

    return (image, edge), label

# =========================================================
# DATASET
# =========================================================

def prepare_dataset(path, shuffle):

    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(image_size, image_size),
        batch_size=batch_size,
        label_mode='categorical',
        shuffle=shuffle
    )

    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

train_ds = prepare_dataset(f"{BASE}/train", True)

val_ds = prepare_dataset(f"{BASE}/valid", False)

test_ds = prepare_dataset(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [ ]:
def create_dual_model():

    # RGB input
    rgb_input = layers.Input(
        shape=(image_size, image_size, 3),
        name="rgb_input"
    )

    base = EfficientNetV2S(
        include_top=False,
        weights='imagenet',
        input_tensor=rgb_input
    )

    base.trainable = True

    # Freeze early layers
    for layer in base.layers[:-50]:
        layer.trainable = False

    # Middle feature map
    middle_layer = base.get_layer("block4c_add").output

    # =====================================================
    # EDGE BRANCH
    # =====================================================

    edge_input = layers.Input(
        shape=(image_size, image_size, 1),
        name="edge_input"
    )

    x = layers.Conv2D(
        32,
        3,
        activation='relu',
        padding='same'
    )(edge_input)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(
        64,
        3,
        activation='relu',
        padding='same'
    )(x)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(
        128,
        3,
        activation='relu',
        padding='same'
    )(x)

    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(
        256,
        3,
        activation='relu',
        padding='same'
    )(x)

    # Resize
    target_size = middle_layer.shape[1:3]

    x = layers.Resizing(
        target_size[0],
        target_size[1]
    )(x)

    # =====================================================
    # FUSION
    # =====================================================

    fused = layers.Concatenate()([middle_layer, x])

    # =====================================================
    # POST FUSION CNN
    # =====================================================

    fused = layers.Conv2D(
        256,
        3,
        activation='relu',
        padding='same'
    )(fused)

    fused = layers.BatchNormalization()(fused)

    fused = layers.MaxPooling2D(2)(fused)

    fused = layers.Dropout(0.3)(fused)

    fused = layers.Conv2D(
        512,
        3,
        activation='relu',
        padding='same'
    )(fused)

    fused = layers.BatchNormalization()(fused)

    fused = layers.GlobalAveragePooling2D()(fused)

    fused = layers.Dropout(0.5)(fused)

    outputs = layers.Dense(
        num_classes,
        activation='softmax',
        dtype='float32'
    )(fused)

    model = tf.keras.Model(
        inputs=[rgb_input, edge_input],
        outputs=outputs
    )

    return model

In [ ]:
model = create_dual_model()

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ rgb_input           │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 256, 256,  │          0 │ rgb_input[0][0]   │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 128, 128,  │        648 │ rescaling[0][0]   │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 128, 128,  │         96 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 128, 128,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 128, 128,  │      5,184 │ stem_activation[… │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 128, 128,  │         96 │ block1a_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 128, 128,  │          0 │ block1a_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_add (Add)   │ (None, 128, 128,  │          0 │ block1a_project_… │
│                     │ 24)               │            │ stem_activation[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_co… │ (None, 128, 128,  │      5,184 │ block1a_add[0][0] │
│ (Conv2D)            │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_bn  │ (None, 128, 128,  │         96 │ block1b_project_… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_project_ac… │ (None, 128, 128,  │          0 │ block1b_project_… │
│ (Activation)        │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_drop        │ (None, 128, 128,  │          0 │ block1b_project_… │
│ (Dropout)           │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1b_add (Add)   │ (None, 128, 128,  │          0 │ block1b_drop[0][… │
│                     │ 24)               │            │ block1a_add[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 64, 64,    │     20,736 │ block1b_add[0][0] │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 64, 64,    │        384 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 64, 64,    │          0 │ block2a_expand_b

 Total params: 3,774,970 (14.40 MB)

 Trainable params: 2,455,554 (9.37 MB)

 Non-trainable params: 1,319,416 (5.03 MB)

In [ ]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(1e-4),

    loss=tf.keras.losses.CategoricalCrossentropy(
        label_smoothing=0.05
    ),

    metrics=['accuracy']
)

In [ ]:
checkpoint_best = ModelCheckpoint(
    filepath=f"{CHECKPOINT_DIR}/best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
print("\n==============================")
print("TRAINING")
print("==============================\n")

history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=15,

    callbacks=[
        checkpoint_best,
        reduce_lr,
        early_stop
    ]
)


TRAINING

Epoch 1/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.6893 - loss: 0.5999
Epoch 1: val_accuracy improved from None to 0.79620, saving model to /drive/MyDrive/checkpoints/best_model.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_model.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 243s 433ms/step - accuracy: 0.7755 - loss: 0.5103 - val_accuracy: 0.7962 - val_loss: 0.4518 - learning_rate: 1.0000e-04
Epoch 2/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - accuracy: 0.8837 - loss: 0.3543
Epoch 2: val_accuracy improved from 0.79620 to 0.90010, saving model to /drive/MyDrive/checkpoints/best_model.keras

Epoch 2: finished saving model to /drive/MyDrive/checkpoints/best_model.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 188s 374ms/step - accuracy: 0.8907 - loss: 0.3406 - val_accuracy: 0.9001 - val_loss: 0.3105 - learning_rate: 1.0000e-04
Epoch 3/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - accuracy: 0.9031 - loss: 0.3194
Epoch 3: val_accuracy improved from 0.90

In [ ]:
model = tf.keras.models.load_model(
    f"{CHECKPOINT_DIR}/best_model.keras"
)

In [ ]:
print("\n==============================")
print("TEST EVALUATION")
print("==============================\n")

loss, acc = model.evaluate(test_ds)

print(f"\nFinal Test Accuracy: {acc:.4f}")


TEST EVALUATION

63/63 ━━━━━━━━━━━━━━━━━━━━ 33s 425ms/step - accuracy: 0.9142 - loss: 0.2841

Final Test Accuracy: 0.9142


In [ ]:
model.save(MODEL_SAVE_PATH)

print("\n==============================")
print("MODEL SAVED SUCCESSFULLY")
print("==============================")


MODEL SAVED SUCCESSFULLY
